In [ ]:
CORS 문제 교과서 수준 분석

1장: CORS 기본 개념

SAME-ORIGIN POLICY란 무엇인가? 

Same Origin Policy는 브라우저의 핵심 보안 메커니즘입니다. 
이 정책은 한 출처에서 로드된 문서나 스크립트가 다른 출처의 리소스와 상호작용하는 것을 제한합니다.
Origin의 구성 요소

Protocol http://, https://
도메인
포트

예시
https://kotlip.kr/api/video/upload/status

프로토콜: https
도메인: kotlip.kr
포트: 443

Same-Origin 판단
https://kotlip.kr -> Same0Origin
https://kotlip.kr -> https://proxy1.aiserv.ktcloud.com:10285 Corss-Origin

Cross-Origin 요청이 필요한 이유

현대 웹 애플리케이션은 여러 서버와 통신합니다.

프론트 엔드 서버와
API 서버와
프론트엔드에서 API 서버로 직접 요청을 보내려면 Cross-Origin 요청이 필요합니다. 

CORS가 해결하는 문제
CORS는 브라우저가 Cross-Origin 요청을 안정하게 허용할 수 잇또록 하는 문제입니다. 

프론트엔드 kotlip.kr
API 서버 kotlip.kr에서 온 요청을 허용합니다. CORs 헤더 포함

브라우저가 요청 허용

브라우저가 응답 헤더를 검사

결과 처리

Access-Control-Allow-Origin: *

모든 출처에서 온 요청을 허용합니다.

단순 요청에서만 사용 가능
credential 모드 (credentials: include)와 함께 사용 불가
인증 정보가 필요한 요청과 함께 사용 불가

예시: 와일드 카드 사용 가능

단순 요청 Simple Request에서만 사용 가능
credentials 모드 (credentials: include)와 함께 사용 불가
인증 정보 (쿠키, Authorization 헤더가) 필요한 요청과 함께 사용 불가

예시 = 와일드 카드 사용 가능

fetch"https://api.example.com/data

fetch https://api.example.com/data
credentials: include

특정 origin 지정 방식 

형식: 

Access-Control-Allow-Origin: https://kotlip.kr
의미: 오직 https://kotlip.kr에서만 허용합니다.

장점: 

credentials 모드와 함께 사용 가능
보안성 향상
인증 정보 전송 가능

동적 origin 허용 서버는 요청 Origin 헤더를 확인하여 동적으로 허용할 origin을 결정할 수 있습니다.

요청 헤더:
Origin: https://kotlip.kr
서버 응답

Access-Control-Aloow-Origin
Acess-Control-Allow-Credentials의 역할
credentials 모드 (credentials: 'include')란

credentials 모드는 Cross-Origin 요청 시 쿠키와 인증 정보를 함께 전송하는 방식입니다.

fetch ... credentials: "include" 

전송되는 정보:
쿠키 (Cookies)
HTTP 인증 정보 (Authorization HEADER)
클라이언트 인증서 

쿠키, 인증 헤더 전송과의 관계

credentials 없이

요청:

GET /api/data HTTP/1.1
Origin: https://kotlip.kr

credentials: include ... 

두 헤더의 상호작용 규칙

왈일드 카드와 credentials의 화환성 문제 

CORS 정책 규칙: 

Access Control Allow Origin: 과 Access Control Allow Credentials: true는 동시에 사용할 수 없습니다. 
Access-control-allow-origin access-controll-allow-creedentials true는 동시에 사용할 수 ㅇ벗습니다
와일드 카드는 *은 모든 출처 허용을 의미
credentials는 인증 정보 전송을 의미
보안상 모든 출처에 인증 정보를 허용하는 것은 위험 

$http_origin 변수의 의미
요청 헤더에서 Origin 추출
nginx의 $http_origin 변수는 클라이언트가 보낸 요청 헤더의 Origin 값을 자동으로 추출합니다.

요청 예시:

GET /api/video/upload/status HTTP/1.1
Host: kotlip.kr
Origin: https://kotlip.kr 이 값을 $http_progin이 추출!

nginx 설정
add_header Access-Control-Allow-Origin $http_origin always;
nginx가 응답에 Access

proxy_hidde_header와 add_header의 동작 

백엔드 헤더 숨기기 

nginx 설정: 

proxy_hide_header Access-Control-Allow-Origin;
proxy_hide_header Access-Control-Allow-Credentials;

의미: 

백엔드가 보낸 Access-Control-Allow-Origin 헤더를 응답에서 제거
백엔드가 보낸 Access-Control-Allow-Credentials 헤더를 응답에서 제거 

이유: 
백엔드가 Access-Control-Allow-Origin: * 을 보내는 경우
nginx가 이를 숨기고 올바른 헤더로 교체해야함. 

흐름: 

백엔드 응답

Access-Control-Allow-Origin: *
Access-Control-Allow-Credentials: true

nginx 처리
proxy_hide_header로 위 헤더 제거한다. 
add_header로 새 헤더 추가한다. 
Access-Controll-Allow-Origin: https://kotlip.kr
Access-Control-Allow-Credentials: true
add_header Access-Control-Allow-Credentials ture always;

add_header: 응답에 새 헤더 추가
$http_origin: 요청의 Origin 값을 사용
always: 성공/에러 응답 모두에 헤더 추가

always 플래그의 역할: 
기본적으로 

Preflight  요청 OPTIONS 처리

Preflight 요청은 브라우저가 실제 요청 전에 서버의 CORS 정책을 확인하는 사전 요청입닌다. 

Preflight가 발생하는 조건 

복잡한 요청 (Complex Request)

PUT, DELETE, PATCH 메서드 

캐스텀 헤더 사용

content-Type 이 application/json 등

Preflight 요청 예시

OPTION /api/video/upload/status HTTP/1.1
Origin: https://kotlip.kr
Access-Control-Request-Method:GET
Access-Control-Request-Headers: authorization

if ($request_method = OPTIONS) {
    return 204;
}

HTTP/1.1 204 No content
Access-Control-Allow-Origin: https://kotlip.kr
Access-Control-Allow-Credentials: true
Access-Control-AllowMethods: GET 

브라우저 검증: 

